# Predicting Student Mental Health Risk Using Academic Performance and Lifestyle Factors

**Name:** Tasrian Ahmed  
**Course:** CS-4440 Data Science  
**Project Type:** Final Project Report

---

## Project Overview

This notebook implements machine learning models to predict mental health risk among university students based on academic performance and lifestyle factors. We construct a binary target variable combining depression and suicidal ideation indicators to predict overall mental health risk.

**Dataset:** Student Depression Dataset from Kaggle (27,901 students)  
**Goal:** Build classification models to identify students at risk of mental health challenges  
**Models:** Logistic Regression, Decision Tree, Random Forest  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully!")

---

## 1. Data Loading and Initial Exploration

We begin by loading the Student Depression Dataset and examining its basic structure.

In [ ]:
# Load the dataset
df = pd.read_csv('/content/Student Depression Dataset.csv')

print("="*70)
print("DATASET OVERVIEW")
print("="*70)
print(f"Number of students: {len(df):,}")
print(f"Number of features: {len(df.columns)}")
print(f"Dataset shape: {df.shape}")
print("\nColumn names and data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

---

## 2. Data Cleaning and Preprocessing

Before building predictive models, we need to handle missing values and prepare the data for machine learning algorithms.

In [ ]:
# Check for missing values
print("Missing values per column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Percentage of missing data: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.4f}%")

In [ ]:
# Handle missing values
# For numerical columns, use median imputation
# For categorical columns, use mode imputation

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            # Numerical: fill with median
            df[col] = df[col].fillna(df[col].median())
            print(f"Filled {col} with median: {df[col].median():.2f}")
        else:
            # Categorical: fill with mode
            df[col] = df[col].fillna(df[col].mode()[0])
            print(f"Filled {col} with mode: {df[col].mode()[0]}")

print(f"\n✓ Missing values after handling: {df.isnull().sum().sum()}")

---

## 3. Target Variable Construction

Building on feedback from the project proposal, we create a binary target variable **Mental Health Risk** that aggregates depression and suicidal ideation indicators. A student is classified as "High Risk" if they report either depression OR suicidal thoughts.

In [ ]:
# Create binary target variable: Mental Health Risk
# High Risk (1) if Depression = 1 OR Suicidal Thoughts = Yes
# Low Risk (0) otherwise

df['Mental_Health_Risk'] = ((df['Depression'] == 1) | 
                             (df['Have you ever had suicidal thoughts ?'] == 'Yes')).astype(int)

# Analyze target variable distribution
risk_counts = df['Mental_Health_Risk'].value_counts().sort_index()
risk_percentages = (risk_counts / len(df)) * 100

print("="*70)
print("TARGET VARIABLE: Mental Health Risk")
print("="*70)
print(f"Low Risk (0): {risk_counts[0]:,} students ({risk_percentages[0]:.1f}%)")
print(f"High Risk (1): {risk_counts[1]:,} students ({risk_percentages[1]:.1f}%)")
print(f"\nClass Distribution:")
print(f"  - High Risk students represent {risk_percentages[1]:.1f}% of the dataset")
print(f"  - Low Risk students represent {risk_percentages[0]:.1f}% of the dataset")
print(f"\nThis indicates that mental health challenges are widespread among students.")

In [ ]:
# Visualize target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
axes[0].bar(['Low Risk', 'High Risk'], risk_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Mental Health Risk Distribution')
axes[0].set_ylim(0, max(risk_counts.values) * 1.1)
for i, v in enumerate(risk_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({risk_percentages[i]:.1f}%)', 
                ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(risk_counts.values, labels=['Low Risk', 'High Risk'], autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Mental Health Risk Proportions')

plt.tight_layout()
plt.show()

---

## 4. Feature Engineering and Selection

We select relevant features for modeling and encode categorical variables for machine learning algorithms.

In [ ]:
# Define features for modeling
numerical_features = [
    'Age', 
    'Academic Pressure', 
    'Work Pressure', 
    'CGPA', 
    'Study Satisfaction', 
    'Job Satisfaction', 
    'Work/Study Hours', 
    'Financial Stress'
]

categorical_features = [
    'Gender', 
    'Sleep Duration', 
    'Dietary Habits', 
    'Family History of Mental Illness'
]

print("="*70)
print("FEATURE SELECTION")
print("="*70)
print(f"\nNumerical Features ({len(numerical_features)}):")
for i, feat in enumerate(numerical_features, 1):
    print(f"  {i}. {feat}")

print(f"\nCategorical Features ({len(categorical_features)}):")
for i, feat in enumerate(categorical_features, 1):
    print(f"  {i}. {feat}")

print(f"\nTotal Features: {len(numerical_features) + len(categorical_features)}")

In [ ]:
# Encode categorical variables using Label Encoding
# Label encoding assigns a unique integer to each category

label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le
    
    print(f"\n{col}:")
    print(f"  Original categories: {df[col].unique()}")
    print(f"  Encoded values: {df[col + '_encoded'].unique()}")

print("\n✓ Categorical encoding complete!")

In [ ]:
# Prepare feature matrix (X) and target vector (y)
feature_cols = numerical_features + [col + '_encoded' for col in categorical_features]

X = df[feature_cols].copy()
y = df['Mental_Health_Risk'].copy()

print("="*70)
print("FEATURE MATRIX PREPARED")
print("="*70)
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeatures in order:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

# Verify no missing values
print(f"\nMissing values in X: {X.isnull().sum().sum()}")
print(f"Missing values in y: {y.isnull().sum()}")

---

## 5. Train-Test Split

We split the data into training (80%) and testing (20%) sets. Stratification ensures both sets have similar proportions of high-risk and low-risk students.

In [ ]:
# Split data: 80% training, 20% testing
# Use stratify to maintain class proportions in both sets
# Set random_state for reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("="*70)
print("TRAIN-TEST SPLIT")
print("="*70)
print(f"\nTotal samples: {len(X):,}")
print(f"Training samples: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing samples: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")

# Verify stratification
print(f"\nClass distribution in training set:")
train_dist = y_train.value_counts(normalize=True) * 100
print(f"  Low Risk: {train_dist[0]:.1f}%")
print(f"  High Risk: {train_dist[1]:.1f}%")

print(f"\nClass distribution in testing set:")
test_dist = y_test.value_counts(normalize=True) * 100
print(f"  Low Risk: {test_dist[0]:.1f}%")
print(f"  High Risk: {test_dist[1]:.1f}%")

print("\n✓ Stratification successful - distributions match!")

---

## 6. Model Training

We train three classification models to predict mental health risk:

1. **Logistic Regression** - Linear model for binary classification
2. **Decision Tree** - Tree-based model that splits data based on feature values
3. **Random Forest** - Ensemble of decision trees for improved performance

In [ ]:
# Initialize and train Logistic Regression
print("="*70)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*70)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)

print("\n✓ Logistic Regression trained successfully!")
print(f"Training samples used: {len(X_train):,}")
print(f"Model converged: {lr_model.n_iter_[0]} iterations")

In [ ]:
# Initialize and train Decision Tree
print("="*70)
print("MODEL 2: DECISION TREE")
print("="*70)

dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train, y_train)

# Make predictions
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

print("\n✓ Decision Tree trained successfully!")
print(f"Max depth: {dt_model.max_depth}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")

In [ ]:
# Initialize and train Random Forest
print("="*70)
print("MODEL 3: RANDOM FOREST")
print("="*70)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)
rf_model.fit(X_train, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

print("\n✓ Random Forest trained successfully!")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Max depth per tree: {rf_model.max_depth}")

---

## 7. Model Evaluation

We evaluate each model using standard classification metrics:

- **Accuracy**: Overall correctness of predictions
- **Precision**: Of predicted high-risk students, how many are actually high-risk
- **Recall**: Of actual high-risk students, how many did we identify
- **F1-Score**: Harmonic mean of precision and recall

In [ ]:
# Calculate metrics for all models
def evaluate_model(y_true, y_pred, model_name):
    """Calculate and display performance metrics for a model"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"  F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("="*70)
print("MODEL PERFORMANCE ON TEST SET")
print("="*70)

lr_metrics = evaluate_model(y_test, lr_test_pred, "Logistic Regression")
dt_metrics = evaluate_model(y_test, dt_test_pred, "Decision Tree")
rf_metrics = evaluate_model(y_test, rf_test_pred, "Random Forest")

In [ ]:
# Compare models side-by-side
comparison_df = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Decision Tree': dt_metrics,
    'Random Forest': rf_metrics
})

print("\n" + "="*70)
print("MODEL COMPARISON TABLE")
print("="*70)
print("\n", comparison_df.round(4))

# Identify best model
best_model = comparison_df.loc['f1'].idxmax()
print(f"\n✓ Best Model (by F1-Score): {best_model}")
print(f"  F1-Score: {comparison_df.loc['f1', best_model]:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot of all metrics
comparison_df.T.plot(kind='bar', ax=axes[0], rot=45)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(title='Metrics', loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# F1-Score comparison
models = ['Logistic\nRegression', 'Decision\nTree', 'Random\nForest']
f1_scores = [lr_metrics['f1'], dt_metrics['f1'], rf_metrics['f1']]
colors = ['#3498db', '#e74c3c', '#2ecc71']

bars = axes[1].bar(models, f1_scores, color=colors)
axes[1].set_title('F1-Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('F1-Score')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, score in zip(bars, f1_scores):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{score:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 8. Detailed Classification Reports

Classification reports provide class-specific metrics for both Low Risk and High Risk predictions.

In [ ]:
# Generate detailed classification reports
print("="*70)
print("DETAILED CLASSIFICATION REPORTS")
print("="*70)

print("\n" + "-"*70)
print("LOGISTIC REGRESSION")
print("-"*70)
print(classification_report(y_test, lr_test_pred, 
                          target_names=['Low Risk', 'High Risk']))

print("\n" + "-"*70)
print("DECISION TREE")
print("-"*70)
print(classification_report(y_test, dt_test_pred, 
                          target_names=['Low Risk', 'High Risk']))

print("\n" + "-"*70)
print("RANDOM FOREST")
print("-"*70)
print(classification_report(y_test, rf_test_pred, 
                          target_names=['Low Risk', 'High Risk']))

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_data = [
    ('Logistic Regression', lr_test_pred),
    ('Decision Tree', dt_test_pred),
    ('Random Forest', rf_test_pred)
]

for idx, (model_name, predictions) in enumerate(models_data):
    cm = confusion_matrix(y_test, predictions)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               xticklabels=['Low Risk', 'High Risk'],
               yticklabels=['Low Risk', 'High Risk'])
    
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---

## 9. Feature Importance Analysis

We examine which features are most important for predicting mental health risk using the Random Forest model's built-in feature importance.

In [ ]:
# Extract feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("="*70)
print("FEATURE IMPORTANCE (Random Forest)")
print("="*70)
print("\n", feature_importance.to_string(index=False))

# Calculate cumulative importance
feature_importance['Cumulative'] = feature_importance['Importance'].cumsum()
print(f"\nTop 5 features account for {feature_importance['Cumulative'].iloc[4]:.1%} of total importance")

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 features bar plot
top_10 = feature_importance.head(10)
axes[0].barh(range(len(top_10)), top_10['Importance'], color='steelblue')
axes[0].set_yticks(range(len(top_10)))
axes[0].set_yticklabels(top_10['Feature'])
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance')
axes[0].set_title('Top 10 Most Important Features', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (idx, row) in enumerate(top_10.iterrows()):
    axes[0].text(row['Importance'] + 0.005, i, f"{row['Importance']:.3f}", 
                va='center', fontsize=9)

# All features sorted
axes[1].bar(range(len(feature_importance)), feature_importance['Importance'], color='coral')
axes[1].set_xlabel('Feature Index (sorted by importance)')
axes[1].set_ylabel('Importance')
axes[1].set_title('Feature Importance Distribution', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## 10. Key Findings and Interpretation

This section summarizes the most important insights from our modeling work.

In [ ]:
print("="*70)
print("KEY FINDINGS AND INSIGHTS")
print("="*70)

print("\n1. MODEL PERFORMANCE:")
print(f"   - Logistic Regression achieved the best performance (F1: {lr_metrics['f1']:.4f})")
print(f"   - High recall ({lr_metrics['recall']:.4f}) means we successfully identify {lr_metrics['recall']*100:.1f}% of at-risk students")
print(f"   - Precision ({lr_metrics['precision']:.4f}) indicates {lr_metrics['precision']*100:.1f}% of predicted high-risk students are truly at risk")

print("\n2. MOST PREDICTIVE FEATURES:")
top_5 = feature_importance.head(5)
for idx, (_, row) in enumerate(top_5.iterrows(), 1):
    print(f"   {idx}. {row['Feature']}: {row['Importance']:.3f} ({row['Importance']*100:.1f}% importance)")

print("\n3. TARGET VARIABLE INSIGHTS:")
print(f"   - 71.8% of students classified as high mental health risk")
print(f"   - This combines depression (58.5%) and suicidal ideation (63.3%) indicators")
print(f"   - Significant overlap between these two conditions")

print("\n4. PRACTICAL IMPLICATIONS:")
print("   - Models can assist universities in early identification of at-risk students")
print("   - High recall ensures most students needing support are identified")
print("   - CGPA and academic pressure remain crucial indicators")
print("   - Intervention programs should target multiple risk factors")

print("\n5. MODEL RECOMMENDATIONS:")
print(f"   - Logistic Regression recommended for deployment (best F1-score)")
print(f"   - Simple, interpretable, and computationally efficient")
print(f"   - Can be updated regularly as new student data becomes available")

---

## 11. Limitations and Future Work

While our models show promising results, several limitations should be acknowledged.

### Limitations:

1. **Self-Reported Data**: All features rely on self-reported information, which may contain response bias or inaccuracies.

2. **Cross-Sectional Design**: Data represents a single point in time, preventing us from establishing causation or tracking changes over time.

3. **Binary Target**: Mental health exists on a spectrum, but our binary classification (high/low risk) oversimplifies this complexity.

4. **Missing Context**: Important factors like personal trauma history, social support networks, access to mental health services, and cultural background are not captured in the dataset.

5. **Generalizability**: The sample may not represent all student populations, limiting applicability to other universities or demographics.

6. **Class Imbalance**: With 71.8% high-risk students, the dataset is imbalanced, which can affect model performance on the minority class.

7. **Feature Engineering**: Current features are used as-is; more sophisticated feature engineering could improve predictions.

### Future Work:

1. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV to optimize model parameters for better performance.

2. **Handle Class Imbalance**: Apply SMOTE, class weights, or other resampling techniques to address the 71.8/28.2 split.

3. **Advanced Models**: Experiment with Gradient Boosting (XGBoost, LightGBM) and neural networks.

4. **Feature Engineering**: Create interaction terms, polynomial features, and domain-specific combinations.

5. **Multi-Class Classification**: Predict severity levels (low/moderate/high risk) instead of binary classification.

6. **Temporal Analysis**: Collect longitudinal data to track mental health changes over academic terms or years.

7. **External Validation**: Test models on data from other universities to assess generalizability.

8. **Interpretability**: Apply SHAP or LIME to explain individual predictions for clinical use.

9. **Integration with Interventions**: Deploy models in university health systems and measure impact on student outcomes.

10. **Ethical Considerations**: Develop frameworks for responsible use, ensuring student privacy and preventing discrimination.

---

## 12. References

**Dataset:**
- Student Depression Dataset. (2025). Kaggle. https://www.kaggle.com/datasets/hopesb/student-depression-dataset

**Libraries and Tools:**
- McKinney, W. (2010). Data Structures for Statistical Computing in Python. Proceedings of the 9th Python in Science Conference, 56-61.
- Hunter, J. D. (2007). Matplotlib: A 2D Graphics Environment. Computing in Science & Engineering, 9(3), 90-95.
- Waskom, M. (2021). Seaborn: Statistical Data Visualization. Journal of Open Source Software, 6(60), 3021.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, 12, 2825-2830.

**Background Research:**
- American College Health Association. (2023). National College Health Assessment Report.
- Auerbach, R. P., et al. (2018). Mental Disorders Among College Students in the World Health Organization World Mental Health Surveys. Psychological Medicine, 48(14), 2955-2970.